# 🚀 Market Strategy: Data Intelligence for a New Literary Horizon

## Business Context and Opportunity
The rise of COVID-19 reshaped global consumption habits. With physical spaces closed, the home became the epicenter of entertainment, driving demand for digital reading platforms. In this environment of accelerated growth, **startups** don't just compete on volume — they compete on the precision of their offering. This analysis audits a key competitor's database to extract the foundations of a **winning value proposition** in the post-pandemic market.

## Project Roadmap

To reach a 360° view of the business, this study is structured around the following strategic pillars:

* **Pillar 0: Infrastructure and Data Validation:** Establishing a robust, reliable environment and mapping the data structure.
* **Pillar 1: Currency and Freshness Audit:** Analyzing the volume of publications in the digital era (post-2000) to measure how fresh the catalog is against new market demands.
* **Pillar 2: Quality and Popularity Diagnosis:** Cross-referencing quantitative (ratings) and qualitative (reviews) metrics to identify the "anchor" titles that drive user satisfaction.
* **Pillar 3: Supply Intelligence (Publishers):** Evaluating publisher efficiency to identify suppliers that deliver not just volume, but high engagement per title.
* **Pillar 4: Author Prestige and Brand Insurance:** Identifying authors with the strongest statistical reputation to minimize inventory investment risk.
* **Pillar 5: The Human Factor and Social Capital:** Analyzing "Power Users" to understand who drives social proof and platform credibility through their reviews.
* **Pillar 6: Conclusions and Strategic Roadmap:** Synthesizing findings and business proposals for launching a new, competitive literary product.

---

## 🛠️ Pillar 0: Infrastructure and Data Validation
*(Setting up the analysis environment and integrity audit)*


Before extracting strategic value, we need to make sure our "raw material" (the data) is complete and reliable. This **Technical Setup** phase acts as the quality control step that ensures every downstream finding is trustworthy enough to inform investment decisions.


**Tech Stack Components:**
* **Orchestration Engine:** `SQLAlchemy` for session management and communication with the PostgreSQL dialect.
* **Security Layer:** `sslmode='require'` parameter to mitigate the risk of data interception.
* **Data Interface:** `Pandas` as the transformation engine to convert relational schemas into workable tabular structures.

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine

# Load environment variables from the .env file
load_dotenv()

# Retrieve credentials using os.getenv
# If not found in .env, fall back to default values
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'db': os.getenv('DB_NAME'),
}

# Build the connection string cleanly
connection_string = f"postgresql://{db_config['user']}:{db_config['pwd']}@{db_config['host']}:{db_config['port']}/{db_config['db']}"

# Create the connection engine
engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

#### **Integrity Validation**
Once the infrastructure is defined, we move on to validating connectivity. A fundamental pillar of good data governance is confirming the data channel is active before running heavy read operations — this avoids execution errors later in the notebook.

In [2]:
# Validating the integrity of the communication channel
try:
    # Attempt to establish a physical connection and run a minimal query
    with engine.connect() as connection:
        print("✅ Connection established successfully.")
        print(f"Status: SQLAlchemy engine is linked to database: {db_config['db']}")
except Exception as e:
    print("❌ Connection error: Could not establish a link with the server.")
    print(f"Technical detail: {e}")

✅ Connection established successfully.
Status: SQLAlchemy engine is linked to database: data-analyst-final-project-db


### 0.1 Information Inventory Audit
For a startup to operate with agility, we start with a **data ecosystem mapping** to identify what assets are available to us. 

This step is the equivalent of doing a physical inventory in a warehouse: we confirm we have inventory records (Books), suppliers (Publishers), creators (Authors), and — most importantly in this stay-at-home context — the voice of the customer (Reviews and Ratings).

**What do we get out of this?**
* **Full Transparency:** We identify the official sources of information.
* **Scope Assurance:** We confirm we have all the "pieces of the puzzle" needed to answer the business questions.

In [3]:
# Query the tables that exist in the public schema
query_schema = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
ORDER BY table_name;
"""

df_tables = pd.io.sql.read_sql(query_schema, con=engine)
print("Tables found in the database:")
display(df_tables)

Tables found in the database:


,table_name
0,advertisment_costs
1,authors
2,books
3,check_avg
4,orders
5,publishers
6,ratings
7,reviews
8,visits


### Data Inventory Interpretation

Querying the public schema revealed an ecosystem of **9 entities**, confirming this is a robust database covering not just the catalog, but also platform operations and marketing. 

For the purposes of our **Strategic Analysis**, we'll classify the tables into three functional dimensions:

1. **Product Dimension (Catalog):**
   * `books`, `authors`, and `publishers`: Hold the structural information about the works and their creators.
2. **Interaction Dimension (Voice of the Customer):**
   * `ratings` and `reviews`: Critical data for measuring title satisfaction and prestige.
3. **Operations and Marketing Dimension (Effectiveness):**
   * `advertisment_costs`, `orders`, `visits`, and `check_avg`: These tables suggest the database supports **ROI (Return on Investment)** and purchasing behavior analysis, going well beyond a simple library catalog.

**Technical Note:** We identified tables like `check_avg` and `visits` that could be key to correlating traffic with perceived book quality. However, to stay aligned with the project's initial objectives, we'll prioritize the intersection between the **Product** and **Interaction** dimensions.

---

### 0.2 Sanity Check: Asset Quality Validation

Now that we know what tables exist, we run an **Entity Sampling** pass. This isn't just a technical step — it's a business safeguard. By inspecting the first records, we confirm that the data captured during the pandemic activity spike is coherent and usable.

**Goals of this inspection:**
1. **Operational Consistency:** Confirm date and number formats are correct so our "most popular books" calculations come out accurate.
2. **Information Bridges:** Identify the connection points (IDs) that will let us cross user behavior with publisher success.
3. **Signal Cleanup:** Visually spot any missing data that could bias our conclusions.

*Below is a quick look at the 5 pillars of information that will support our value proposition:*

In [4]:
# Table exploration
tables = ['books', 'authors', 'publishers', 'ratings', 'reviews']

for table in tables:
    print(f"--- Exploring table: {table.upper()} ---")

    # Load table
    query = f"SELECT * FROM {table};"
    df_temp = pd.io.sql.read_sql(query, con=engine)

    # Preview
    display(df_temp.head(5))

    # Shape summary
    print(f"📐 Shape: {df_temp.shape[0]:,} rows x {df_temp.shape[1]} columns")

    # Duplicate audit
    duplicates = df_temp.duplicated().sum()
    print(f"👯 Exact duplicate rows: {duplicates}")

    # Null value audit
    nulls = df_temp.isnull().sum()
    nulls_with_values = nulls[nulls > 0]

    if not nulls_with_values.empty:
        print("⚠️ Null values by column:")
        for col, count in nulls_with_values.items():
            pct = (count / len(df_temp)) * 100
            print(f"   - {col}: {count:,} ({pct:.2f}%)")
    else:
        print("✅ No null values.")

    print("\n" + "=" * 50 + "\n")

--- Exploring table: BOOKS ---


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


📐 Shape: 1,000 rows x 6 columns
👯 Exact duplicate rows: 0
✅ No null values.


--- Exploring table: AUTHORS ---


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


📐 Shape: 636 rows x 2 columns
👯 Exact duplicate rows: 0
✅ No null values.


--- Exploring table: PUBLISHERS ---


...

📐 Shape: 340 rows x 2 columns
👯 Exact duplicate rows: 0
✅ No null values.


--- Exploring table: RATINGS ---


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


📐 Shape: 6,456 rows x 4 columns
👯 Exact duplicate rows: 0
✅ No null values.


--- Exploring table: REVIEWS ---


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


📐 Shape: 2,793 rows x 4 columns
👯 Exact duplicate rows: 0
✅ No null values.




### Exploration Findings and Relational Mapping

After a granular inspection of the records, we've validated the structure needed to move forward with the indicator analysis. Key findings below:

#### 1. Catalog Structure (Master Entities)
* **`BOOKS`**: This is the pivot table. It holds critical attributes like `num_pages` (integer) and `publication_date` (DATE format), which will let us segment by content volume and time period.
* **`AUTHORS` & `PUBLISHERS`**: Act as dimension tables. Note that the `authors` table includes collaborations (e.g., authors separated by slashes), which is a factor to account for when aggregating data.

#### 2. Interaction Dynamics (Voice of the Customer)
* **`RATINGS`**: Provides the quantitative metric (`rating`). Note that users can rate the same book (e.g., `book_id: 1` rated by both 'ryanfranco' and 'grantpatricia'), which means we need to use averaging functions (`AVG`).
* **`REVIEWS`**: Adds the qualitative layer. The overlap in `book_id` and `username` between this table and `ratings` confirms we can cross-reference the data to check whether longer reviews correlate with higher ratings.

#### 3. Relational Connectivity (Foreign Keys)
The **Relational Ecosystem** is confirmed:
* `book_id` connects the catalog to customer satisfaction (`ratings` and `reviews`).
* `author_id` and `publisher_id` link the works to their creators and distributors.


**Pillar Conclusion:** The database is normalized and the data is consistent with the project's objectives.

---


## 📅 Pillar 1: Currency and Freshness Audit
*(Analyzing catalog freshness in the post-2000 era)*


For a new product to break through successfully, the first step is dissecting the competitor's catalog. We're not looking to copy — we're looking to identify **gaps and opportunities** in their inventory. The goal of this pillar is to measure how "up to date" the competitor's value proposition is, and whether their offering matches the demand from readers that emerged during lockdown.

### 1.1 Freshness Analysis: The Competitor's "Timestamp"
How modern is the service we're trying to outdo? We evaluate the volume of their catalog published from **January 1, 2000** onward. 

**Why this data is gold for our Value Proposition:**
* **Detecting "Zombie Inventory":** If the competitor has a large catalog but few modern (post-2000) titles, our startup can differentiate itself by positioning as the "latest trends" platform.
* **Fit with the Digital-First User:** Users who adopted new apps during the pandemic are looking for fresh content. Knowing how many recent titles the competitor offers lets us fine-tune our licensing strategy to outpace them on relevance.

**Why is this insight critical for the startup?**
* **Competitive Positioning:** Determines whether our value proposition should lean toward classic collecting or current digital consumption trends.
* **Audience Segmentation:** A younger catalog attracts users who are more active on tech platforms — the same users who were the main early adopters of new apps during lockdown.

In [5]:
# Query to compare the total universe vs. recent content
query_universe = """
SELECT 
    COUNT(book_id) AS total_books,
    COUNT(CASE WHEN publication_date > '2000-01-01' THEN 1 END) AS recent_books,
    ROUND(COUNT(CASE WHEN publication_date >= '2000-01-01' THEN 1 END) * 100.0 / COUNT(book_id), 2) AS percentage_recent
FROM books;
"""

df_universe = pd.io.sql.read_sql(query_universe, con=engine)
print(f"Strategic Finding: The competitor offers {df_universe.iloc[0,1]} titles from the modern era.")
display(df_universe)

Strategic Finding: The competitor offers 819 titles from the modern era.


,total_books,recent_books,percentage_recent
0,1000,819,82.1


### 💡 Freshness Analysis: The Competitor's Inventory Profile

Finding **821 titles** published from January 1, 2000 onward reveals the competitor's market thesis. Comparing this figure against their total universe (1,000 books), we can draw critical conclusions for our roadmap:

* **Dominance of the Contemporary Market:** The competitor has bet heavily on currency, with **82.1% of their catalog concentrated in the digital era**. This tells us their value proposition is built on "newness." For our startup, going head-to-head on this volume would require an aggressive investment in recent licenses.
* **The Classics "Blind Spot":** Only **17.9%** of their offering is pre-millennium content. If post-pandemic market analysis points to a resurgence of interest in classics or backlist reading, there's an **opportunity gap** here where we could differentiate without needing to compete for the same titles.
* **Rotation Agility:** Such a young catalog suggests the competitor has acquisition processes optimized for 21st-century trends. This means our own technical infrastructure needs to be extremely agile to keep pace in the race for new releases.



**Strategic Impact for Development:** The competitor is "over-indexed" on modernity. Our winning value proposition should validate whether this 82% of modern books actually drives conversation (reviews); otherwise, we'd be looking at a "filler" catalog where we could win by offering **quality over quantity**.

## 💎 Pillar 2: Quality and Popularity Diagnosis
*(Identifying "anchor" titles through quantitative and qualitative metrics)*


Once catalog freshness is validated, **developing the proposal** requires dissecting the performance of the competitor's assets. It's not enough to know what books they offer — we need to know which ones are the "real engines" of their platform, and which are just noise in the inventory.

### 2.1 Asset Audit: Mass Popularity vs. Real Quality

For this **development** phase, we designed an audit metric that breaks down the competitor's engagement into two critical dimensions: **Quantitative (Ratings)** and **Qualitative (Text Reviews)**. 

**Why does this distinction matter for our market strategy?**
In the digital environment, a book can rack up hundreds of "quick clicks" (star ratings) with very little effort, but only titles that spark written reviews build a loyal community. By contrasting average satisfaction with participation volume, we measure the **solidity of the competitor's prestige**:

* **Risk Validation:** Identifying "fragile" rating averages (few votes) lets us avoid the mistakes the competitor is making and come in with statistically better-validated titles.
* **Detecting Deep Engagement:** Books with a high density of text reviews are the assets that actually build post-pandemic user loyalty. Our proposal needs to capture this level of active conversation to beat out the competitor's colder system.

This audit lets us map which of the competitor's books are "noise" and which are the real pillars of their success — the ones we should emulate or surpass.

In [6]:
# SQL query to pull the entire catalog with aggregated metrics
query_metrics_full = """
WITH book_ratings AS (
    SELECT 
        book_id,
        COUNT(rating_id) AS total_ratings,
        ROUND(AVG(rating), 2) AS avg_rating,
        MIN(rating) AS min_rating,
        MAX(rating) AS max_rating
    FROM ratings
    GROUP BY book_id
),
book_reviews AS (
    SELECT 
        book_id,
        COUNT(review_id) AS total_reviews
    FROM reviews
    GROUP BY book_id
)
SELECT 
    b.book_id,
    b.title AS "Title",
    COALESCE(r.total_ratings, 0) AS "Total Ratings",
    COALESCE(r.avg_rating, 0.00) AS "Average Rating",
    COALESCE(r.min_rating, 0) AS "Min Rating",
    COALESCE(r.max_rating, 0) AS "Max Rating",
    COALESCE(v.total_reviews, 0) AS "Total Text Reviews"
FROM books AS b
LEFT JOIN book_ratings AS r ON b.book_id = r.book_id
LEFT JOIN book_reviews AS v ON b.book_id = v.book_id
ORDER BY "Total Ratings" DESC;
"""

# Load the full dataset
df_books_full = pd.io.sql.read_sql(query_metrics_full, con=engine)

# Display the complete dataset
print(f"Full Dataset Shape: {df_books_full.shape[0]} rows × {df_books_full.shape[1]} columns")
print("Overall sample of the full inventory:")

# Displaying df_books_full directly (or adjusting pd.set_option if desired)
# shows the overall structure of the full catalog.
display(df_books_full)

Full Dataset Shape: 1000 rows × 7 columns
Overall sample of the full inventory:


,book_id,Title,Total Ratings,Average Rating,Min Rating,Max Rating,Total Text Reviews
0,948,Twilight (Twilight #1),160,3.66,1,5,7
1,750,The Hobbit or There and Back Again,88,4.13,2,5,6
2,673,The Catcher in the Rye,86,3.83,2,5,6
3,75,Angels & Demons (Robert Langdon #1),84,3.68,1,5,5
4,302,Harry Potter and the Prisoner of Azkaban (Harr...,82,4.41,2,5,6
...,...,...,...,...,...,...,...
995,465,Naked Empire (Sword of Truth #8),2,3.50,3,4,1
996,55,A Woman of Substance (Emma Harte Saga #1),2,5.00,5,5,2
997,652,The Body in the Library (Miss Marple #3),2,4.50,4,5,2
998,790,The Magicians' Guild (Black Magician Trilogy #1),2,3.50,3,4,2


In [7]:
# Building the derived view: Top 10 Books
df_top10 = df_books_full.head(10).copy()

# Assign an explicit ranking to differentiate this table from the full dataset
df_top10.insert(0, 'Ranking', range(1, 11))

print("--- DERIVED ANALYSIS: TOP 10 BOOKS BY ENGAGEMENT ---")

# Clean, direct display
display(df_top10.reset_index(drop=True))

--- DERIVED ANALYSIS: TOP 10 BOOKS BY ENGAGEMENT ---


,Ranking,book_id,Title,Total Ratings,Average Rating,Min Rating,Max Rating,Total Text Reviews
0,1,948,Twilight (Twilight #1),160,3.66,1,5,7
1,2,750,The Hobbit or There and Back Again,88,4.13,2,5,6
2,3,673,The Catcher in the Rye,86,3.83,2,5,6
3,4,75,Angels & Demons (Robert Langdon #1),84,3.68,1,5,5
4,5,302,Harry Potter and the Prisoner of Azkaban (Harr...,82,4.41,2,5,6
5,6,299,Harry Potter and the Chamber of Secrets (Harry...,80,4.29,1,5,6
6,7,301,Harry Potter and the Order of the Phoenix (Har...,75,4.19,2,5,5
7,8,722,The Fellowship of the Ring (The Lord of the Ri...,74,4.39,2,5,5
8,9,79,Animal Farm,74,3.73,2,5,5
9,10,300,Harry Potter and the Half-Blood Prince (Harry ...,73,4.25,2,5,5


### 📊 Pillar 2 Interpretation: Catalog Diagnosis

#### 1. Observed Findings (Data-Driven)
* **Volume Distribution:** The full catalog (1,000 books) was processed. Twilight leads total quantitative engagement with **160 ratings**, followed by *The Hobbit* (**88**) and *The Catcher in the Rye* (**86**).
* **Rating Quality:** Within the most-engaged set, *Harry Potter and the Prisoner of Azkaban* posts the highest average rating (**4.41**), while *Twilight* sits at an average of **3.66**.
* **Ratings vs. Text Reviews:** There's a considerable gap between star ratings and written reviews. For example, *Twilight* has 160 ratings but only **7 text reviews**; *The Hobbit* has 88 ratings and **6 text reviews**.

---

#### 2. Business Hypotheses (Require Further Validation)
* **Retention Hypothesis:** The gap between high quantitative ratings and low written review counts could suggest the competitor's community is predominantly passive. *Note: This hypothesis would need to be tested against retention rate or in-app time-spent data.*
* **Acquisition vs. Satisfaction Strategy:** While titles like *Twilight* work as mass volume drivers, their lower average rating could point to a long-term user experience risk if not paired with personalized recommendations.

## 🏭 Pillar 3: Supply Intelligence (Publishers)
*(Evaluating publisher efficiency and engagement per supplier)*

For this **proposal development**, it's essential to identify the business partners that sustain the competitor's content infrastructure. Not every supplier delivers the same value, so we audit who dominates the "deep reading" segment (works > 50 pages), filtering out pamphlets or ephemeral material that doesn't drive long-term retention.

### 3.1 Identifying High-Volume Suppliers

This analysis lets us map the negotiating leverage and dependency the competitor has with major publishing houses. By identifying the volume leaders in substantial content, we can chart our own rights-acquisition strategy.

**Strategic goals for development:**
* **Partnership Mapping:** Who does the competitor have their strongest contracts with? 
* **Barrier-to-Entry Detection:** If one publisher controls 40% of the competitor's catalog, our proposal needs to look for alternatives or negotiate with that supplier's rivals to differentiate our offering.
* **Inventory Quality:** Filtering for books over 50 pages ensures we're analyzing real editorial assets, not promotional content.

In [8]:
# Query to find publishers with books over 50 pages
query_publishers = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS total_books
FROM publishers AS p
JOIN books AS b 
    ON p.publisher_id = b.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_books DESC
LIMIT 10;
"""

df_publishers = pd.io.sql.read_sql(query_publishers, con=engine)
display(df_publishers)

,publisher,total_books
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Ballantine Books,19
5,Bantam,19
6,Berkley,17
7,St. Martin's Press,14
8,Berkley Books,14
9,William Morrow Paperbacks,13


### 💡 Supply Diagnosis: Market Concentration

Analyzing the competitor's top 10 suppliers reveals a highly concentrated inventory structure:

1. **Penguin's Dominance:** With **42 high-impact titles**, Penguin Books isn't just the leader — it's the central axis of the competitor's offering. For **proposal development**, this is either a "must-have" supplier or a direct competitor to neutralize with exclusives from other imprints.
2. **Premium Segment (Vintage and Grand Central):** The presence of these imprints (31 and 25 books respectively) shows the competitor has a solid base of prestige literature. 
3. **Risk Diversification:** The fact that the Top 10 is made up of internationally established names suggests the competitor prefers the safety of established brands over independent content.

**Impact for Development:** We've identified that the competitor's "muscle" depends on fewer than a dozen major contracts. For our value proposition to be disruptive, we could explore deals with fast-growing independent publishers the competitor is ignoring, offering a fresh, exclusive reading alternative that breaks the major-imprint monopoly identified here.

### 3.2 Publisher Efficiency Analysis: Volume vs. Engagement

For **proposal development**, knowing the size of the competitor's inventory isn't enough. We need to uncover the real relationship between their catalog presence (*market share*) and the traction their titles generate (*engagement*). 

In this phase, we calculate the **Engagement Ratio**, averaging the reviews and ratings earned per book published under each imprint. This KPI is key to distinguishing between:
1. **Volume Suppliers:** Add "bulk" to the catalog but with moderate interaction.
2. **Impact Suppliers:** Imprints that, regardless of size, keep an extremely active reader community.

**Why is this critical for development?**
Spotting where the competitor has high volume but low impact (low ratio) reveals inefficient assets. Conversely, high ratios point to publishers our proposal should prioritize to guarantee a vibrant user base from day one.

In [9]:
# SQL query for the Publisher Analysis pillar
query_publisher_ratio = """
WITH publisher_metrics AS (
    SELECT 
        b.publisher_id,
        p.publisher AS publisher_name,
        COUNT(DISTINCT b.book_id) AS total_books,
        COUNT(r.rating_id) AS total_ratings,
        COUNT(v.review_id) AS total_reviews
    FROM books AS b
    LEFT JOIN publishers AS p ON b.publisher_id = p.publisher_id
    LEFT JOIN ratings AS r ON b.book_id = r.book_id
    LEFT JOIN reviews AS v ON b.book_id = v.book_id
    GROUP BY b.publisher_id, p.publisher
    -- Minimum volume filter to avoid bias from single-book publishers (optional, per project criteria)
    HAVING COUNT(DISTINCT b.book_id) >= 5 
)
SELECT 
    publisher_name AS "Publisher",
    total_books AS "Total Books",
    total_ratings AS "Total Ratings",
    total_reviews AS "Total Reviews",
    -- Explicit calculation of ratings per book
    ROUND(total_ratings::numeric / total_books, 2) AS "Ratings per Book",
    -- Calculation of reviews per book
    ROUND(total_reviews::numeric / total_books, 2) AS "Reviews per Book"
FROM publisher_metrics
-- KEY FIX: Sort by the ratio/efficiency metric, NOT by book volume
ORDER BY "Ratings per Book" DESC;
"""

df_publisher_ratio = pd.io.sql.read_sql(query_publisher_ratio, con=engine)

print("--- PUBLISHER EFFICIENCY ANALYSIS (SORTED BY RATIO) ---")
display(df_publisher_ratio.head(10))

--- PUBLISHER EFFICIENCY ANALYSIS (SORTED BY RATIO) ---


,Publisher,Total Books,Total Ratings,Total Reviews,Ratings per Book,Reviews per Book
0,Little Brown and Company,12,1225,1225,102.08,102.08
1,NAL,5,398,398,79.60,79.60
2,Back Bay Books,11,773,773,70.27,70.27
3,Riverhead Books,5,321,321,64.20,64.20
4,Pocket Books,10,516,516,51.60,51.60
5,Signet Classics,6,301,301,50.17,50.17
6,Anchor Books,9,436,436,48.44,48.44
7,Anchor,9,396,396,44.00,44.00
8,Dell Publishing Company,8,325,325,40.63,40.63
9,Penguin Books,42,1571,1571,37.40,37.40


### 💡 Efficiency Interpretation: Who Drives the Highest Engagement Density?

Sorting the results in descending order by **Ratings per Book**, the publisher hierarchy reveals decisive conclusions for our catalog strategy:

#### 1. Engagement Density Leader: Little Brown and Company
Unlike the absolute-volume analysis, when evaluating efficiency per title, **Little Brown and Company** tops the table with a ratio of **102.08** ratings/reviews per book (1,225 interactions spread across just 12 titles).
* **Insight for Development:** This is a catalog with a highly active community. Focusing curation on deals or titles similar to this imprint guarantees strong response and interaction on the platform.

#### 2. Density in Mid-Sized Catalogs: NAL and Back Bay Books
Publishers with moderate output show a proportionally much stronger response than the mass-market publishers:
* **NAL:** Posts a ratio of **79.60** interactions per book with only 5 titles in catalog.
* **Back Bay Books:** Reaches a ratio of **70.27** interactions per title across its 11 books.
* **Insight for Development:** This shows a massive catalog isn't required to generate high engagement volume. These imprints represent pockets of high participation that could be replicated in our proposal.

#### 3. Mass-Volume Performance: The Penguin Books Case
While **Penguin Books** is the competitor's highest-volume supplier by far (42 books and 1,571 ratings), in terms of density per title it ranks **#10** with a ratio of **37.40**.
* **Development Strategy:** Their massive catalog dilutes the average engagement density per book. They represent a broad coverage pillar, but need to be paired with high-density imprints to keep the platform dynamic.

---

### 🎯 Pillar 3 Conclusion: Publisher Diagnosis

A publisher's efficiency in this context is measured by its ability to generate social engagement density (ratings and reviews) per title published. 

While the competitor's offering spans mass-volume imprints like **Penguin Books** (whose average density per book is lower), our strategy can outperform the competitor by prioritizing and promoting high-impact imprints like **Little Brown and Company** or **NAL**, which achieve significantly higher participation levels with a much smaller title count.

> **Methodological Note:** These metrics strictly reflect interaction density recorded in the app (ratings and reviews per book) and are not direct indicators of sales, commercial conversion, inventory turnover, or financial profitability.

## ✒️ Pillar 4: Author Prestige and Brand Insurance
*(Validating statistical reputation to reduce risk)*

For **proposal development**, identifying the figures behind the competitor's credibility is a "brand insurance" step. In this phase, we're looking to spot authors who don't just have fame, but an **undeniable statistical reputation**. 

### 4.1 Identifying Authorship Assets (Minimum 50 Ratings)

In this analysis, we answer a key question: *Which of the competitor's authors have the highest-rated books under a representative sample?* To avoid bias from inflated averages built on very few votes, we apply a **50-Rating Relevance Filter**.

**Strategic value for development:**
* **Satisfaction Guarantee:** We identify the works the market has already validated as "excellent."
* **Risk Reduction:** Selecting authors with these indicators for our proposal minimizes the risk of user rejection.
* **Quality Benchmark:** We establish the rating standard (e.g., 4.41) our flagship content needs to hit to stay competitive.

In [10]:
# Author with the highest average rating
query_top_author = """
WITH valid_books AS (
    -- Filtering books with AT LEAST 50 ratings (>= 50)
    SELECT 
        book_id,
        COUNT(rating_id) AS total_ratings
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)
-- 2. Grouping by author, considering only their valid books
SELECT 
    a.author AS "Author",
    COUNT(DISTINCT b.book_id) AS "Rated Books (>=50)",
    SUM(vb.total_ratings) AS "Total Ratings",
    ROUND(AVG(r.rating), 2) AS "Author Average Rating"
FROM valid_books AS vb
JOIN books AS b ON vb.book_id = b.book_id
JOIN authors AS a ON b.author_id = a.author_id
JOIN ratings AS r ON b.book_id = r.book_id
GROUP BY a.author_id, a.author
ORDER BY "Author Average Rating" DESC
LIMIT 10;
"""

df_top_author = pd.io.sql.read_sql(query_top_author, con=engine)

print("--- TOP AUTHORS BY AVERAGE RATING (BOOKS WITH >= 50 RATINGS) ---")
display(df_top_author)

--- TOP AUTHORS BY AVERAGE RATING (BOOKS WITH >= 50 RATINGS) ---


,Author,Rated Books (>=50),Total Ratings,Author Average Rating
0,J.K. Rowling/Mary GrandPré,4,24078.0,4.29
1,Markus Zusak/Cao Xuân Việt Khương,1,2809.0,4.26
2,J.R.R. Tolkien,2,13220.0,4.25
3,Louisa May Alcott,1,2704.0,4.19
4,Rick Riordan,1,3844.0,4.08
5,William Golding,1,5041.0,3.90
6,J.D. Salinger,1,7396.0,3.83
7,William Shakespeare/Paul Werstine/Barbara A. M...,1,4356.0,3.79
8,Paulo Coelho/Alan R. Clarke/Özdemir İnce,1,3249.0,3.79
9,Lois Lowry,1,3136.0,3.75


### 📊 Pillar 4 Interpretation: Identifying the Leading Author

Applying the strict filter for books with **at least 50 ratings** (`COUNT(rating_id) >= 50`), the author hierarchy reflects the real qualitative perception of active readers on the platform:

#### 1. Observed Findings (Data-Driven)
* **Rating Leader:** **J.K. Rowling/Mary GrandPré** tops the ranking with the highest average rating (**4.29** out of 5), backed by **4 rated books** that clear the 50-rating threshold and a massive combined total of **24,078 ratings**.
* **Consistency Among Popular Authors:** In second and third place are **Markus Zusak/Cao Xuân Việt Khương** with an average of **4.26** (1 book, 2,809 ratings) and **J.R.R. Tolkien** with **4.25** (2 books, 13,220 ratings).
* **Selection Volume:** Only a select group of authors manages to hold averages above 4.00 when their titles accumulate thousands of interactions, with *J.K. Rowling* the only Top 3 author sustaining this rating across a larger title count (4 books).

---

#### 2. Methodological Notes and Analysis Limits
* **Inclusion Criterion (`>= 50`):** Using the `>= 50` threshold ensures statistical representativeness by excluding works with too few ratings that could skew the average, guaranteeing only authors with an established catalog presence are evaluated.
* **Interpretation Limits:** This metric exclusively measures **average qualitative satisfaction and reception** among active in-app users. **It is not an indicator of total sales volume, revenue, or commercial popularity outside the analyzed database.**

---

### 🎯 Pillar 4 Conclusion: Author Diagnosis

Focusing on high-participation books (`>= 50` ratings) confirms that **J.K. Rowling/Mary GrandPré** is the highest-value authorship asset on the platform in qualitative terms, combining the highest average rating (**4.29**) with the largest number of popular works in the Top 10 (4 titles).

For developing the new value proposition, prioritizing commercial deals or featured placement for the Top 3 authors' works (**J.K. Rowling**, **Markus Zusak**, and **J.R.R. Tolkien**) guarantees a proven high-satisfaction offering, maximizing engagement and qualitative retention among the reader community.

---


## 👥 Pillar 5: The Human Factor and Social Capital
*(Analyzing Power Users and the strength of social proof)*

To wrap up the audit, **proposal development** needs to focus on the hardest asset to replicate: the community. We're not analyzing users generically — we apply a **Funnel Analysis** to identify the competitor's "evangelists."

### 5.1 Segmentation Analysis: From Casual User to Power User

This study lets us measure the competitor's social health. A platform can have thousands of users, but its real strength lies in its **Power Users**: the ones whose loyalty and activity sustain the social proof (reviews) that convinces others to read.

**Critical dimensions for development:**
1. **Market Reach:** Total unique users interacting with the competitor.
2. **Identifying the Elite (Power Users):** Users with extreme activity (> 50 ratings).
3. **Qualitative Productivity:** How many text reviews this segment contributes. If this number is high, the competitor has a strong barrier to entry built on user-generated content (UGC).

In [11]:
# Funnel query: Total Users vs Power Users vs Power User Reviews vs Total Review Universe
query_user_funnel = """
WITH power_users AS (
    -- 1. Identify users with more than 50 ratings
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(book_id) > 50
),
power_user_reviews AS (
    -- 2. Count the text reviews written by Power Users
    SELECT 
        username,
        COUNT(review_id) AS review_count
    FROM reviews
    WHERE username IN (SELECT username FROM power_users)
    GROUP BY username
)
SELECT 
    (SELECT COUNT(DISTINCT username) FROM ratings) AS total_users_platform,
    (SELECT COUNT(*) FROM power_users) AS count_power_users,
    ROUND(AVG(review_count), 2) AS avg_text_reviews_power_users,
    -- New: total reviews written by Power Users
    COALESCE(SUM(review_count), 0) AS total_reviews_power_users,
    -- New: total text review universe across the platform
    (SELECT COUNT(*) FROM reviews) AS total_reviews_universe
FROM power_user_reviews;
"""

df_funnel = pd.io.sql.read_sql(query_user_funnel, con=engine)
display(df_funnel)

,total_users_platform,count_power_users,avg_text_reviews_power_users,total_reviews_power_users,total_reviews_universe
0,160,6,24.33,146.0,2793


### 📊 Social Capital Interpretation: Community Diagnosis

The competitor's user funnel results give us key data to understand their community dynamics and support **our proposal development**:

* **Share of Frequent Users:** Out of a universe of **160 users**, only **6 qualify as Power Users** (more than 50 ratings), representing **3.75%** of the total community.
* **Individual Writing Intensity:** These 6 users show high loyalty and a consistent habit, averaging **24.33 text reviews** per person (146 reviews total).
* **Actual Content Distribution:** Against a global universe of **2,793 text reviews**, this group's contribution represents **5.23%** of all written content on the platform. This shows the competitor's community **doesn't depend on a small handful of creators** — content generation is broadly distributed across the remaining 96.25% of users.

---


### 5.2 Identifying "Super Contributors": Frequent User Profile

To wrap up the behavior audit, we run a **nominal inspection** of the segment with the highest number of recorded ratings. 

Having confirmed that this segment represents **3.75%** of the total user base (6 of 160 users) and contributes **5.23%** of all written reviews globally, it's worth analyzing their individual behavior to characterize their reading and interaction habits.

**Analytical goals for the proposal:**
* **Segment Characterization:** Analyze the individual activity of the 6 most active profiles.
* **Average vs. Frequent User Profile:** Understand the behavioral differences between the general community and frequent users to design more effective retention incentives on our own platform.

In [12]:
# Query to identify the 6 Power Users and their review volume
query_top_6_users = """
SELECT 
    username AS "Username",
    COUNT(review_id) AS "Total Text Reviews"
FROM reviews
WHERE 
    username IN (
        SELECT username
        FROM ratings
        GROUP BY username
        HAVING COUNT(book_id) > 50
    )
GROUP BY username
ORDER BY "Total Text Reviews" DESC;
"""

df_top_6 = pd.io.sql.read_sql(query_top_6_users, con=engine)
display(df_top_6)

,Username,Total Text Reviews
0,sfitzgerald,28
1,martinadam,27
2,richard89,26
3,jennifermiller,25
4,paul88,22
5,xdavis,18


### 💡 Interpretation: Nominal Inspection of Frequent Users

The nominal inspection results let us characterize the writing habits of the most active group (the 6 users with more than 50 ratings):

1. **Qualitative Writing Leaders:** Users **sfitzgerald** (28 reviews), **martinadam** (27), **richard89** (26), and **jennifermiller** (25) lead the pack, each generating 25 or more text reviews on the platform.
2. **Segment Activity Range:** Within this *Power User* group, individual output ranges from **28 reviews** for the top contributor (*sfitzgerald*) down to **18 reviews** for the sixth spot (*xdavis*), adding up to a combined total of **146 text reviews** (an average of 24.33 per user).
3. **Strategic Touchpoints:** Although this group represents a small share of overall content (5.23% of the platform's total reviews), their high recurrence points to a highly loyal user profile.

---

### 🎯 Pillar 5 Final Conclusion

Individual inspection of frequent users reveals specific profiles (*sfitzgerald*, *martinadam*, *richard89*, etc.) with an outstanding, consistent habit of qualitative participation. 

For **developing our proposal**, this finding confirms two complementary lines of action:
* **Retention/Loyalty Strategy:** Design recognition programs (profile badges, featured-reviewer status, or advanced editing tools) to engage high-intensity profiles.
* **Community Growth Strategy:** Implement simple feedback mechanics that encourage the large majority of occasional and mid-level users to publish their first text review, raising the platform's overall participation rate.

## 🗺️ Pillar 6: Conclusions and Strategic Roadmap
*(Synthesizing findings and guidelines for proposal development)*

After completing the multidimensional audit of the competing service, we've turned raw data into a business intelligence structure. **Proposal development** now has precise statistical grounding to ensure a balanced, sustainable launch.

---

### 🎯 Competitive Intelligence Synthesis

The findings from each strategic pillar validating the new product's viability are consolidated below:

1. **Currency and Opportunity (Pillar 1):** The competitor runs a predominantly modern catalog, with **82% of content published from the year 2000 onward**.
   * **Conclusion:** Catalog freshness is a market standard. For our proposal, the edge will come from balancing new releases with a strategic curation of the 18% of classic works that still maintain steady engagement.

2. **Asset Diagnosis and View Presentation (Pillar 2):** We structured the full **1,000-book** catalog alongside a selected **Top 10** view, differentiating between mass reach and qualitative engagement.
   * **Conclusion:** Our proposal should use the highest-reach titles for user acquisition, but base its recommendations on titles with high qualitative ratings to maximize reader satisfaction.

3. **Supply Efficiency (Pillar 3):** Ranking publishers by their engagement-per-book ratio (`Total Ratings / Total Books`), **Little Brown and Company** leads participation density with **102.08** interactions per title. In contrast, mass-market catalogs like **Penguin Books** post an average ratio of **37.40** (ranking #10).
   * **Conclusion:** Catalog volume doesn't guarantee proportional engagement. The proposal should prioritize partnerships with high-density imprints to optimize inventory without overloading the platform.

4. **Brand Insurance (Pillar 4):** Evaluating only works with at least 50 ratings (`COUNT(rating_id) >= 50`), **J.K. Rowling/Mary GrandPré** stands out as the author with the highest average rating (**4.29** out of 5), backed by 4 popular books and more than 24,000 ratings.
   * **Conclusion:** Having authors with strong, consolidated reputation and volume acts as the platform's "brand insurance," securing authority from day one.

5. **Social Capital and Community Dynamics (Pillar 5):** We identified **6 Power Users** (3.75% of the total base of 160 users) averaging **24.33 text reviews** per person. Together, they contribute **146 reviews**, representing **5.23%** of the total review universe (2,793).
   * **Conclusion:** The data rules out a monopoly of written content. Most reviews (94.77%) come from occasional and mid-level users, calling for a dual strategy: retain super-active users while encouraging first-time posts from the general community.

---

### 📈 Strategic Conclusion: Boosting Engagement and Gamification

The community behavior diagnosis reveals a core of high-frequency users alongside a broad base of occasional participants. This structure opens a clear opportunity for shaping the value proposition:

* **Gamification and Incentives Strategy:** Design a loyalty program that recognizes both recurrence and qualitative value creation (text reviews). Activities like profile badges, featured-reviewer status, or advanced reading tools can serve as incentive drivers.
* **Activating the General Base:** Since the distributed community base is the primary source of overall reviews (94.77%), simplifying the interface for writing short reviews will increase the conversion rate from passive readers to active contributors.
* **Business Impact (LTV and Retention):** Maintaining a steady flow of organic reviews strengthens the platform's social proof, improves usability, and reduces churn without incurring high acquisition costs.

---

### 🚀 Final Value Proposition

Success in this market comes down to turning data accumulation into **effective community connections**. This study shows that surgical catalog curation, efficient supplier selection, and activating a distributed community are the foundational pillars for building a modern, agile, and highly competitive platform.

---